# Open premises and staffed service in Finnish library territories

Run all cells to reproduce the reported territory comparisons and figures from the deposited analytic inputs. The calculations retain both endpoint periods, both weightings and the source domains for missing hours. They do not reconstruct original provider files.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'checksums.sha256').is_file()), None)
if root is None:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('Open this notebook from the extracted repository folder.')
    print('Upload a repository ZIP containing library-service-trajectories/.')
    uploads = files.upload()
    archives = [name for name in uploads if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload one repository ZIP.')
    destination = Path.cwd() / 'reproduction'
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError('Archive contains an invalid path.')
            if item.filename.startswith('library-service-trajectories/'):
                archive.extract(item, destination)
    root = destination / 'library-service-trajectories'
assert (root / 'checksums.sha256').is_file()
sys.dont_write_bytecode = True
sys.path.insert(0, str(root / 'code'))

## Check the scientific inputs

Install the two calculation and plotting dependencies if needed. The checksum check protects eight stable scientific inputs, including the numerical result references.

In [ ]:
import importlib.util
if any(importlib.util.find_spec(name) is None for name in ['numpy', 'matplotlib']):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements.txt')])
from check import check_inputs, check_results
check_inputs(root)

## Reproduce the reported results

The calculation checks 22 detailed result objects before writing the compact outputs. Source-domain calculations use rational arithmetic. Descriptive projection coefficients use numerical linear algebra with a stated comparison tolerance.

In [ ]:
from reproduce import reproduce
parity = reproduce(root)
check_results(root)

## Figures 1–4

The regenerated figures preserve the reported values, categories and omission rules. Figures 1 and 4 keep Siuntio out of finite point clouds and state its unresolved joint domain. Figure 2 marks unbounded hour totals; Figure 3 uses the observed worked-FTE totals.

In [ ]:
subprocess.check_call([sys.executable, '-B', str(root / 'code/figures.py')])
from IPython.display import SVG, display
for number in range(1, 5):
    display(SVG(filename=str(root / 'figures' / f'figure-{number}.svg')))

## Interpretation and data boundary

The primary population-weighted difference is [9.97, 10.16] percentage points; its equal-territory counterpart is [1.58, 3.58]. The later-baseline population-weighted difference is [−2.98, −2.80]. These describe the fixed territory frame, rather than causal effects or individual access.

Geographic correspondence, prior population references and correction decisions enter as deposited analytic inputs. Their numerical consequences are recomputed; the original workbook parsing and documentary adjudication are not repeated. [Data guide](../data/README.md) and [source terms](../licenses/third-party.md).